# 01 - Collecte des Données (Juin 2025)

## Contexte

On télécharge les fichiers **détaillés** AirBnB (79 colonnes) depuis insideairbnb.com.

Pourquoi les fichiers détaillés et non les fichiers de visualisation ?
- Fichiers **visualisation** : 18 colonnes seulement
- Fichiers **détaillés** : 79 colonnes (scores reviews, infos hôte, bathrooms...)

| Ville | Date | Format |
|-------|------|--------|
| Lyon | 15 juin 2025 | listings.csv.gz |
| Paris | 06 juin 2025 | listings.csv.gz |
| Bordeaux | 15 juin 2025 | listings.csv.gz |

## Étape 1 : Téléchargement via `src/data/collect.py`

`download_all()` pour chaque ville :
1. Crée le dossier `data/raw/<ville>/`
2. Vérifie si le CSV existe déjà (affiche `[SKIP]` si oui)
3. Télécharge le `.gz` avec `urllib.request`
4. Décompresse avec `gzip.open` + `shutil.copyfileobj`
5. Supprime le `.gz` intermédiaire

`sys.path.insert(0, '..')` : permet d'importer notre package `src` depuis `notebook/`.

In [ ]:
import sys, os
# Ajoute AirBnb_Projet/ au chemin Python pour importer src/
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from src.data.collect import download_all
download_all(raw_dir='../data/raw')

## Étape 2 : Aperçu des données brutes

`low_memory=False` : évite un avertissement pandas sur les colonnes à types mixtes.
`.shape` retourne (nb_lignes, nb_colonnes).

In [ ]:
import pandas as pd

CITIES = {'Lyon': 'lyon', 'Paris': 'paris', 'Bordeaux': 'bordeaux'}

for name, key in CITIES.items():
    df = pd.read_csv(f'../data/raw/{key}/listings_detail.csv', low_memory=False)
    print(f'{name:10s} : {df.shape[0]:,} annonces x {df.shape[1]} colonnes')

## Étape 3 : Les 79 colonnes disponibles

`df.columns` retourne un Index de tous les noms de colonnes.
On en gardera seulement **26** au prétraitement — on supprime URLs, textes libres, identifiants inutiles.

In [ ]:
df = pd.read_csv('../data/raw/lyon/listings_detail.csv', low_memory=False)
print(f'Nombre de colonnes : {len(df.columns)}')
print(list(df.columns))

## Étape 4 : Aperçu des premières lignes (Lyon)

`df.head(3)` affiche les 3 premières lignes — on voit les données **brutes** :
- prix sous forme de texte `"$120.00"`
- `host_is_superhost` = `"t"` ou `"f"`
- `bathrooms_text` = `"1 bath"`
→ Tout ça sera nettoyé dans le notebook 02.

In [ ]:
df.head(3)

## Étape 5 : Statistiques basiques

`describe()` calcule count, mean, std, min, quartiles, max.
Permet de détecter des anomalies (ex: minimum_nights = 365).

In [ ]:
for name, key in CITIES.items():
    df = pd.read_csv(f'../data/raw/{key}/listings_detail.csv', low_memory=False)
    print(f'\n=== {name} ===')
    print(df[['price', 'minimum_nights', 'number_of_reviews', 'availability_365']].describe())